# Caso práctico: Operaciones con tablas de datos
1. Análisis de transacciones de una empresa minorista
 Objetivo
Ejecutar el ciclo completo de manipulación de datos sobre una tabla con problemas reales: valores faltantes, formatos erróneos, duplicados, atípicos e inconsistencias categóricas, y terminar con un análisis económico básico de ingresos, mix de productos y satisfacción.
Contenido
| Fase | Bloque | Qué se hace |
| 1 | Exploración y diagnóstico | Carga, tipos, nulos, estadística descriptiva |
| 2 | Limpieza y depuración | Duplicados, texto, nulos, outlier |
| 3 | Transformación | `monto_total`, fechas, discretización |
| 4 | Agregación y resumen | `groupby`, tabla dinámica, satisfacción |
| 5 | Selección, filtrado y muestreo | Filtro multicriterio, muestra estratificada |


1.Exploración y diagnóstico | Carga, tipos, nulos, estadística descriptiva |

In [0]:
import numpy as np
import pandas as pd

In [0]:
Ventas = spark.read.format("delta").table("workspace.electiva.caso_practico_ventas")
# Convertir el DataFrame de Spark a Pandas
Ventas = Ventas.toPandas()
     

In [0]:
Ventas

In [0]:
## mostrando las primeras 10 filas de la tabla 
print(f"Registros: {Ventas.shape[0]}  |  Columnas: {Ventas.shape[1]}")
print("\nPrimeras 10 filas:")
mostrar(Ventas.head(10))

In [0]:

# ultimas 5 filas de la tabla 
print("Últimas 5 filas:")
mostrar(Ventas.tail(5))

In [0]:
#Identificación de tipos de datos
# Se compara el tipo que asignó pandas con el tipo esperado según el diccionario del dataset.

tipos_esperados = {
    "id_transaccion": "texto",
    "fecha": "fecha",
    "id_cliente": "texto",
    "categoria_producto": "texto",
    "cantidad": "entero",
    "precio_unitario": "decimal",
    "metodo_pago": "texto",
    "region": "texto",
    "satisfaccion_cliente": "entero",
}
 
def familia_tipo(dtype):
    """Clasifica un dtype de pandas en: fecha, entero, decimal o texto."""
    s = str(dtype).lower()
    if "datetime" in s:
        return "fecha"
    if "int" in s:
        return "entero"
    if "float" in s:
        return "decimal"
    return "texto"
 
diag_tipos = pd.DataFrame({
    "columna": Ventas.columns,
    "tipo_pandas": [str(t) for t in Ventas.dtypes],
    "tipo_esperado": [tipos_esperados[c] for c in Ventas.columns],
})
diag_tipos["coincide"] = [
    "Sí" if familia_tipo(Ventas[c].dtype) == tipos_esperados[c] else "No"
    for c in Ventas.columns
]
mostrar(diag_tipos)


 **Lectura del diagnóstico:** fecha llega como texto; cantidad y satisfaccion_clientellegan como float64 aunque conceptualmente son enteros. Esto ocurre porque pandas usa NaN (que es un decimal) para representar faltantes, lo que obliga a toda la columna a ser decimal. Se corregirá después de imputar.

In [0]:
# Detección de valores nulos
nulos = pd.DataFrame({
    "nulos": Ventas.isnull().sum(),
    "porcentaje_%": (Ventas.isnull().mean() * 100).round(2),
}).reset_index().rename(columns={"index": "columna"})
 
mostrar(nulos)
print(f"Total de celdas vacías en la tabla: {int(Ventas.isnull().sum().sum())}")

In [0]:
# Estadística descriptiva (cantidad y precio_unitario)
estadisticas = (
    Ventas[["cantidad", "precio_unitario"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(2)
    .reset_index()
    .rename(columns={"index": "estadistico"})
)
mostrar(estadisticas)
 

 **Interpretación:** en precio_unitario la **media** es muy superior a la **mediana** y la desviación estándar es enorme. Esa asimetría es la huella de un valor extremo (el máximo de 9.999). La mediana es un estimador **robusto** (casi no se mueve ante atípicos) y la media no; por eso en datos con outliers conviene reportar ambas.

FASE 2 . Limpieza y depuración (Data Cleaning)

In [0]:
#   Se trabaja sobre una **copia** (`df_limpio`) para conservar el original intacto y poder comparar antes/después.
# orden de las operaciones (importa):** primero duplicados, luego texto, luego atípico y nulos. Si se imputan nulos antes de eliminar duplicados, las medianas y modas se calculan sobre registros repetidos y quedan sesgadas.
Ventas_limpio = Ventas.copy()

In [0]:
# Tratamiento de duplicados
# Se elimina por `id_transaccion`: un identificador debe ser único. En este dataset las filas con ID repetido son además copias exactas, así que ambos criterios coinciden.
filas_antes = len(Ventas_limpio)
 
print("Registros duplicados detectados (se muestran todas las copias):")
mostrar(
   Ventas_limpio[Ventas_limpio.duplicated(subset=["id_transaccion"], keep=False)]
    .sort_values("id_transaccion")
)
 
Ventas_limpio = Ventas_limpio.drop_duplicates(subset=["id_transaccion"], keep="first").reset_index(drop=True)
 
print(f"Filas antes: {filas_antes}  |  Filas después: {len(Ventas_limpio)}  |  Eliminadas: {filas_antes - len(Ventas_limpio)}")
 
 

In [0]:
# Normalización de texto en `metodo_pago`
# "Tarjeta Credito" (sin tilde y sin "de") y "Tarjeta de Crédito" son la misma categoría escrita distinto. Si no se unifican, cualquier `groupby` las trataría como métodos de pago diferentes y fragmentaría los resultados.

print("ANTES:")
mostrar(Ventas_limpio["metodo_pago"].value_counts().rename_axis("metodo_pago").reset_index(name="frecuencia"))
 
variantes = {
    "Tarjeta Credito": "Tarjeta de Crédito",
    "Tarjeta credito": "Tarjeta de Crédito",
    "Tarjeta Crédito": "Tarjeta de Crédito",
}
Ventas_limpio["metodo_pago"] = Ventas_limpio["metodo_pago"].str.strip().replace(variantes)
 
print("DESPUÉS:")
mostrar(Ventas_limpio["metodo_pago"].value_counts().rename_axis("metodo_pago").reset_index(name="frecuencia"))
 
 

2.3 Gestión de valores nulos: cantidad y satisfaccion_cliente

- * cantidad` → mediana.* Variable cuantitativa y sesgada a la derecha (hay compras de hasta 10 unidades); la mediana no se ve arrastrada por esas compras grandes.
- * satisfaccion_cliente → moda.* Es una escala ordinal de 1 a 5: promediar no tiene mucho sentido conceptual y la media podría dar valores como 3,69 que no existen en la escala. La moda sí es un valor válido de la escala.

In [0]:
 #Cantidad: mediana
mediana_cantidad = Ventas_limpio["cantidad"].median()
n_nulos_cant = Ventas_limpio["cantidad"].isnull().sum()
Ventas_limpio["cantidad"] = Ventas_limpio["cantidad"].fillna(mediana_cantidad)
 
# Satisfacción: moda
moda_satisf = Ventas_limpio["satisfaccion_cliente"].mode()[0]
n_nulos_sat = Ventas_limpio["satisfaccion_cliente"].isnull().sum()
Ventas_limpio["satisfaccion_cliente"] = Ventas_limpio["satisfaccion_cliente"].fillna(moda_satisf)
 
# Ahora que no hay NaN, las dos columnas pueden volver a ser enteras
Ventas_limpio["cantidad"] = Ventas_limpio["cantidad"].astype(int)
Ventas_limpio["satisfaccion_cliente"] = Ventas_limpio["satisfaccion_cliente"].astype(int)
 
print(f"cantidad: {n_nulos_cant} nulos reemplazados por la mediana = {mediana_cantidad:.0f}")
print(f"satisfaccion_cliente: {n_nulos_sat} nulos reemplazados por la moda = {moda_satisf}")

 Tratamiento del atípico (precio_unitario = 9999.00) y de los nulos de precio

**Diagnóstico:** 9.999 es 20 veces el siguiente precio más alto del catálogo (500). Además, es un número "redondo" típico de **valor centinela** o error de digitación, no de un precio de mercado.
 **Decisión: imputar con el precio medio de su categoría, en lugar de eliminar la fila.**
 - Eliminar descartaría información válida de la transacción (cantidad, región, método de pago, satisfacción).
- Con solo ~120 filas, cada registro pesa.
 - El precio medio se calcula **excluyendo el atípico**; de lo contrario el 9.999 contaminaría su propio reemplazo.

In [0]:
filtro_outlier = Ventas_limpio["precio_unitario"] == 9999.00
 
print("Transacción atípica localizada:")
mostrar(Ventas_limpio[filtro_outlier])
 
# 1) Convertir el atípico en faltante para que no entre en el cálculo de la media
Ventas_limpio.loc[filtro_outlier, "precio_unitario"] = np.nan
 
# 2) Precio medio por categoría (sin el atípico y sin nulos)
precio_medio_cat = Ventas_limpio.groupby("categoria_producto")["precio_unitario"].mean()
print("Precio medio por categoría (referencia de imputación):")
mostrar(precio_medio_cat.round(2).rename("precio_medio").reset_index())
 
# 3) Imputar el atípico + los nulos originales de precio
n_precio_imputar = Ventas_limpio["precio_unitario"].isnull().sum()
Ventas_limpio["precio_unitario"] = Ventas_limpio["precio_unitario"].fillna(
    Ventas_limpio["categoria_producto"].map(precio_medio_cat)
)
Ventas_limpio["precio_unitario"] = Ventas_limpio["precio_unitario"].round(2)
 
print(f"Precios imputados en total (1 atípico + nulos originales): {n_precio_imputar}")

In [0]:
# Verificación final de la fase de limpieza
assert Ventas_limpio.isnull().sum().sum() == 0, "Todavía quedan valores nulos"
assert Ventas_limpio["id_transaccion"].is_unique, "Todavía hay IDs repetidos"
 
print(f"Tabla limpia: {Ventas_limpio.shape[0]} filas × {Ventas_limpio.shape[1]} columnas")
print("Nulos restantes:", int(Ventas_limpio.isnull().sum().sum()))
print("Precio máximo ahora:", Ventas_limpio["precio_unitario"].max())
 
comparacion = pd.DataFrame({
    "antes (original)": Ventas[["cantidad", "precio_unitario"]].agg(["mean", "median", "std", "max"]).stack(),
    "después (limpio)": Ventas_limpio[["cantidad", "precio_unitario"]].agg(["mean", "median", "std", "max"]).stack(),
}).round(2).reset_index()
comparacion.columns = ["estadistico", "variable", "antes (original)", "después (limpio)"]
mostrar(comparacion)

FASE 3: Transformación e ingeniería de atributos

In [0]:
# Columna derivada: `monto_total`
# Ingreso por transacción = cantidad × precio unitario.
Ventas_limpio["monto_total"] = (Ventas_limpio["cantidad"] * Ventas_limpio["precio_unitario"]).round(2)
mostrar(Ventas_limpio[["id_transaccion", "cantidad", "precio_unitario", "monto_total"]].head(10))

In [0]:
# Conversión temporal
#  Con `fecha` como texto no se puede ordenar cronológicamente ni extraer componentes. Se convierte a `datetime64` y se derivan `dia_semana` (en español) y `mes`.
Ventas_limpio["fecha"] = pd.to_datetime(Ventas_limpio["fecha"], format="%Y-%m-%d")
 
dias_es = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves",
           4: "Viernes", 5: "Sábado", 6: "Domingo"}
Ventas_limpio["dia_semana"] = Ventas_limpio["fecha"].dt.dayofweek.map(dias_es)
Ventas_limpio["mes"] = Ventas_limpio["fecha"].dt.month
 
print("Tipo de 'fecha' ahora:", Ventas_limpio["fecha"].dtype)
print(f"Rango de fechas: {Ventas_limpio['fecha'].min().date()} → {Ventas_limpio['fecha'].max().date()}")
mostrar(Ventas_limpio[["fecha", "dia_semana", "mes"]].head(8))


In [0]:
# Discretización (binning) de `monto_total`
# Reglas de la guía: **Bajo** < 100 · **Medio** entre 100 y 500 (ambos incluidos) · **Alto** > 500.
# Como los intervalos son asimétricos (el 100 y el 500 pertenecen a "Medio"), se usa `np.select`, que respeta exactamente esos límites sin los ajustes artificiales que exigiría `pd.cut`.

condiciones = [
    Ventas_limpio["monto_total"] < 100,
    Ventas_limpio["monto_total"] <= 500,   # llega aquí solo si monto >= 100
]
Ventas_limpio["rango_monto"] = np.select(condiciones, ["Bajo", "Medio"], default="Alto")
 
# Categoría ordenada (Bajo < Medio < Alto) para que las tablas salgan en orden lógico
Ventas_limpio["rango_monto"] = pd.Categorical(
    Ventas_limpio["rango_monto"], categories=["Bajo", "Medio", "Alto"], ordered=True
)
 
dist_rangos = (
    Ventas_limpio["rango_monto"].value_counts().sort_index()
    .rename_axis("rango_monto").reset_index(name="transacciones")
)
dist_rangos["participacion_%"] = (dist_rangos["transacciones"] / len(Ventas_limpio) * 100).round(1)
mostrar(dist_rangos)
 

FASE 4: Agregación y resumen de datos

In [0]:
# Tabla dinámica: ingresos por región (filas) × categoría (columnas)
 
# COMMAND ----------
 
tabla_dinamica = pd.pivot_table(
    Ventas_limpio,
    values="monto_total",
    index="region",
    columns="categoria_producto",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total",
).round(2)
 
mostrar(tabla_dinamica.reset_index())
 

In [0]:
# Análisis de satisfacción por método de pago
satisfaccion_pago = (
    Ventas_limpio.groupby("metodo_pago")["satisfaccion_cliente"]
    .agg(satisfaccion_media="mean", n_transacciones="count")
    .round(2)
    .sort_values("satisfaccion_media", ascending=False)
    .reset_index()
)
mostrar(satisfaccion_pago)

 **Cautela al interpretar:** con ~120 observaciones repartidas en 4 métodos de pago, cada grupo tiene pocas transacciones y las diferencias de medias pueden ser ruido muestral. Para afirmar que un método de pago "genera más satisfacción" haría falta una prueba de hipótesis (por ejemplo ANOVA o Kruskal-Wallis), no solo comparar promedios.

- 

FASE 5: Selección, filtrado y muestreo

In [0]:
# Filtrado multicriterio: región Norte **y** satisfacción ≥ 4
 
# COMMAND ----------
 
Ventas_norte_satisfechos = Ventas_limpio[
    (Ventas_limpio["region"] == "Norte") & (Ventas_limpio["satisfaccion_cliente"] >= 4)
]
 
n_norte = (Ventas_limpio["region"] == "Norte").sum()
print(f"Ventas en Norte: {n_norte}  |  Con satisfacción ≥ 4: {len(Ventas_norte_satisfechos)} "
      f"({len(Ventas_norte_satisfechos) / n_norte * 100:.1f}% del Norte)")
mostrar(Ventas_norte_satisfechos)
 
# COMMAND ----------

In [0]:
# Muestreo estratificado: 10% por región
#  El muestreo aleatorio simple podría, por azar, sobre-representar una región. Al muestrear **dentro de cada región** se conserva la composición geográfica de la población. `random_state=42` fija la semilla para que el resultado sea reproducible.
  
muestra = Ventas_limpio.groupby("region").sample(frac=0.10, random_state=42)
 
print(f"Población: {len(Ventas_limpio)} filas  |  Muestra: {len(muestra)} filas")
 
verificacion = pd.DataFrame({
    "poblacion_n": Ventas_limpio["region"].value_counts(),
    "poblacion_%": (Ventas_limpio["region"].value_counts(normalize=True) * 100).round(1),
    "muestra_n": muestra["region"].value_counts(),
    "muestra_%": (muestra["region"].value_counts(normalize=True) * 100).round(1),
}).reset_index().rename(columns={"index": "region", "region": "region"})
mostrar(verificacion)
 
print("Muestra extraída:")
mostrar(muestra.sort_values("region"))